
# Teacher Regularization v1.2.4

`baseline v2.1`를 기준 backbone으로 사용하고, `DANN v1.0`의 domain adaptation loss를 참고해 teacher regularization을 추가한 notebook입니다.

구성:
- student: baseline v2.1의 `MultiViewBidirectionalCrossAttention`
- teacher target group 1: `physics`
- teacher target group 2: `image_structure`
- auxiliary: `domain_head` with GRL


실행하기 전에 physics_solution_class_feature_selection_v2.0.ipynb를 실행해야합니다

이번 버전(`v1.2.4`)은 `onset + motion(physics group)`만 사용하고 `image/domain loss`는 0으로 비활성화합니다.


In [ ]:
from __future__ import annotations

import os
import sys
import json
import random
import shutil
from dataclasses import dataclass
from itertools import cycle
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.preprocessing import StandardScaler
import timm

SRC_DIR = (Path.cwd() / '../../src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from augmentations import build_default_transforms
from output_paths import allocate_output_paths
from reproducibility import make_generator, seed_everything, seed_worker
from models import (
    EMAConfig,
    ModelEMA,
    MultiViewBidirectionalCrossAttentionConfig,
)

DATA_DIR = (Path.cwd() / '../../data').resolve()
FEATURE_CSV = (Path.cwd() / '../../outputs/physics_feature_analysis_v2/class_analysis_features.csv').resolve()
MOTION_CSV = (Path.cwd() / '../../data/motion_targets.csv').resolve()
DEV_MOTION_CSV = (Path.cwd() / '../../outputs/eda_preprocessing/dev_motion_targets.csv').resolve()
assert DATA_DIR.exists(), f'data dir not found: {DATA_DIR}'
assert FEATURE_CSV.exists(), f'feature csv not found: {FEATURE_CSV}'
assert MOTION_CSV.exists(), f'motion csv not found: {MOTION_CSV}'
print('DATA_DIR:', DATA_DIR)
print('FEATURE_CSV:', FEATURE_CSV)
print('MOTION_CSV:', MOTION_CSV)

MOTION_FEATURES = ['max_diff_first', 'mean_diff_prev']
ONSET_CLASS_FEATURE = 'onset_bucket'
IMAGE_STRUCTURE_FEATURES = [
    'abs_delta_structure_center_x',
    'top_structure_center_x',
    'front_structure_center_x',
    'mean_structure_center_x',
    'mean_structure_bbox_w',
]
ALL_AUX_TARGET_COLUMNS = MOTION_FEATURES + [ONSET_CLASS_FEATURE] + IMAGE_STRUCTURE_FEATURES

CFG = {
    'IMG_SIZE': 224,
    'EPOCHS': 100,
    'LEARNING_RATE': 3e-4,
    'BATCH_SIZE': 16,
    'SEED': 42,
    'NUM_WORKERS': 8,
    'WEIGHT_DECAY': 1e-4,
    'MIN_LR': 1e-6,
    'EMA_DECAY': 0.999,
    'EMA_USE_FOR_EVAL': True,
    'EARLY_STOPPING_PATIENCE': 5,
    'MIXUP_ALPHA': 0.1,
    'MIXUP_PROB': 0.0,
    'VIDEO_AUG_ENABLE': True,
    'VIDEO_AUG_CACHE': True,
    'UNSTABLE_START_MIN_SEC': 0.5,
    'UNSTABLE_START_MAX_SEC': 1.0,
    'UNSTABLE_FRAMES_MIN': 2,
    'UNSTABLE_FRAMES_MAX': 3,
    'STABLE_END_MIN_SEC': 9.0,
    'STABLE_END_MAX_SEC': 10.0,
    'STABLE_FRAMES_PER_VIDEO': 2,
    'BACKBONE_NAME': 'convnext_small.fb_in22k_ft_in1k',
    'ATTN_DIM': 256,
    'NUM_HEADS': 8,
    'NUM_LAYERS': 2,
    'POS_GRID': 7,
    'DROPOUT': 0.1,
    'CLASSIFIER_HIDDEN_DIM': 512,
    'CLASSIFIER_MID_DIM': 128,
    'CLASSIFIER_DROPOUT': 0.3,
    'DOMAIN_HIDDEN_DIM': 256,
    'DOMAIN_DROPOUT': 0.2,
    'MOTION_LOSS_WEIGHT': 0.10,
    'ONSET_LOSS_WEIGHT': 0.05,
    'IMAGE_LOSS_WEIGHT': 0.15,
    'DOMAIN_LOSS_WEIGHT': 0.2,
    'GRL_WARMUP_EPOCHS': 5,
    'GRL_MAX_LAMBDA': 0.2,
    'GRL_GAMMA': 4.0,
}

seed_everything(CFG['SEED'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


DATA_DIR: /media/hdd0/whyz/structure-stability/data
FEATURE_CSV: /media/hdd0/whyz/structure-stability/outputs/physics_feature_analysis_v2/class_analysis_features.csv
MOTION_CSV: /media/hdd0/whyz/structure-stability/data/motion_targets.csv
device: cuda


In [2]:
train_df = pd.read_csv(DATA_DIR / 'train.csv', encoding='utf-8-sig')
dev_df = pd.read_csv(DATA_DIR / 'dev.csv', encoding='utf-8-sig')
test_df = pd.read_csv(DATA_DIR / 'sample_submission.csv', encoding='utf-8-sig')
feature_df = pd.read_csv(FEATURE_CSV, encoding='utf-8-sig')

feature_df = feature_df[['split', 'sample_id'] + IMAGE_STRUCTURE_FEATURES].copy()
feature_df = feature_df.rename(columns={'sample_id': 'id'})

for split_name, split_df in [('train', train_df), ('dev', dev_df)]:
    part = feature_df[feature_df['split'].eq(split_name)].drop(columns=['split']).copy()
    missing = set(split_df['id']) - set(part['id'])
    print(split_name, 'image teacher rows:', len(part), 'missing:', len(missing))

image_scaler = StandardScaler()
train_img_mask = feature_df['split'].eq('train') & feature_df[IMAGE_STRUCTURE_FEATURES].notna().all(axis=1)
image_scaler.fit(feature_df.loc[train_img_mask, IMAGE_STRUCTURE_FEATURES])

all_img_mask = feature_df[IMAGE_STRUCTURE_FEATURES].notna().all(axis=1)
feature_df.loc[all_img_mask, IMAGE_STRUCTURE_FEATURES] = image_scaler.transform(feature_df.loc[all_img_mask, IMAGE_STRUCTURE_FEATURES])

print('image teacher normalization: StandardScaler fitted on train split')
print('image normalized rows:', int(all_img_mask.sum()))

motion_df = pd.read_csv(MOTION_CSV, encoding='utf-8-sig')
train_motion_df = motion_df[['id'] + [c for c in MOTION_FEATURES if c in motion_df.columns]].copy()
for c in MOTION_FEATURES:
    if c not in train_motion_df.columns:
        train_motion_df[c] = np.nan

if ONSET_CLASS_FEATURE in motion_df.columns:
    train_motion_df[ONSET_CLASS_FEATURE] = motion_df[ONSET_CLASS_FEATURE]
elif set(['first_move_thr2', 'first_move_thr5']).issubset(motion_df.columns):
    def _onset_bucket(first_move_thr2, first_move_thr5):
        if first_move_thr2 <= 3:
            return 0
        if first_move_thr2 <= 10:
            return 1
        if first_move_thr5 <= 30:
            return 2
        return 3
    train_motion_df[ONSET_CLASS_FEATURE] = [
        _onset_bucket(a, b) if np.isfinite(a) and np.isfinite(b) else -1
        for a, b in zip(motion_df['first_move_thr2'], motion_df['first_move_thr5'])
    ]
else:
    train_motion_df[ONSET_CLASS_FEATURE] = -1


def _onset_bucket_from_video(first_move_thr2, first_move_thr5):
    if first_move_thr2 <= 3:
        return 0
    if first_move_thr2 <= 10:
        return 1
    if first_move_thr5 <= 30:
        return 2
    return 3


def extract_motion_targets_for_split(df, split_dir):
    rows = []
    for row in tqdm(df.itertuples(index=False), total=len(df), desc=f'motion {Path(split_dir).name}', dynamic_ncols=True, ascii=True):
        video_path = Path(split_dir) / row.id / 'simulation.mp4'
        if not video_path.exists():
            rows.append({'id': row.id, **{col: np.nan for col in MOTION_FEATURES}, ONSET_CLASS_FEATURE: -1})
            continue
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            rows.append({'id': row.id, **{col: np.nan for col in MOTION_FEATURES}, ONSET_CLASS_FEATURE: -1})
            continue

        prev = None
        first = None
        diffs_first = []
        diffs_prev = []
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
            if first is None:
                first = gray
            diffs_first.append(float(np.mean(np.abs(gray - first))) * 255.0)
            if prev is None:
                diffs_prev.append(0.0)
            else:
                diffs_prev.append(float(np.mean(np.abs(gray - prev))))
            prev = gray
        cap.release()

        arr_first = np.array(diffs_first, dtype=np.float32)
        max_diff_first = float(arr_first.max()) if arr_first.size else np.nan
        mean_diff_prev = float(np.mean(diffs_prev)) if diffs_prev else np.nan
        idx_thr2 = np.where(arr_first >= 2.0)[0] if arr_first.size else np.array([])
        idx_thr5 = np.where(arr_first >= 5.0)[0] if arr_first.size else np.array([])
        first_move_thr2 = int(idx_thr2[0]) if idx_thr2.size else 999
        first_move_thr5 = int(idx_thr5[0]) if idx_thr5.size else 999

        rows.append({
            'id': row.id,
            'max_diff_first': max_diff_first,
            'mean_diff_prev': mean_diff_prev,
            ONSET_CLASS_FEATURE: _onset_bucket_from_video(first_move_thr2, first_move_thr5) if np.isfinite(max_diff_first) else -1,
        })
    return pd.DataFrame(rows)


regenerate_dev_motion = True
if DEV_MOTION_CSV.exists():
    dev_motion_df = pd.read_csv(DEV_MOTION_CSV, encoding='utf-8-sig')
    required_cols = set(['id'] + MOTION_FEATURES + [ONSET_CLASS_FEATURE])
    if required_cols.issubset(dev_motion_df.columns):
        valid_dev_motion = dev_motion_df[MOTION_FEATURES].notna().all(axis=1).sum()
        if int(valid_dev_motion) > 0:
            regenerate_dev_motion = False
            print('using cached dev motion targets:', DEV_MOTION_CSV, 'valid rows:', int(valid_dev_motion))
if regenerate_dev_motion:
    DEV_MOTION_CSV.parent.mkdir(parents=True, exist_ok=True)
    dev_motion_df = extract_motion_targets_for_split(dev_df[['id', 'label']], DATA_DIR / 'dev')
    dev_motion_df.to_csv(DEV_MOTION_CSV, index=False, encoding='utf-8-sig')
    print('recomputed dev motion targets:', DEV_MOTION_CSV)

motion_scaler = StandardScaler()
train_motion_mask = train_motion_df[MOTION_FEATURES].notna().all(axis=1)
motion_scaler.fit(train_motion_df.loc[train_motion_mask, MOTION_FEATURES])
for part in [train_motion_df, dev_motion_df]:
    mask = part[MOTION_FEATURES].notna().all(axis=1)
    if mask.any():
        part.loc[mask, MOTION_FEATURES] = motion_scaler.transform(part.loc[mask, MOTION_FEATURES])
print('motion teacher normalization: StandardScaler fitted on train split')
print('train motion rows:', len(train_motion_df), 'dev motion rows:', len(dev_motion_df))

train_df = train_df.merge(feature_df[feature_df['split'].eq('train')].drop(columns=['split']), on='id', how='left')
dev_df = dev_df.merge(feature_df[feature_df['split'].eq('dev')].drop(columns=['split']), on='id', how='left')
train_df = train_df.merge(train_motion_df[['id'] + MOTION_FEATURES + [ONSET_CLASS_FEATURE]], on='id', how='left')
dev_df = dev_df.merge(dev_motion_df[['id'] + MOTION_FEATURES + [ONSET_CLASS_FEATURE]], on='id', how='left')
test_df['sample_dir'] = str(DATA_DIR / 'test')

print('train:', train_df.shape)
print('dev:', dev_df.shape)


train image teacher rows: 1000 missing: 0
dev image teacher rows: 100 missing: 0
image teacher normalization: StandardScaler fitted on train split
image normalized rows: 1100


motion dev: 100%|##########| 100/100 [00:00<00:00, 104674.42it/s]

recomputed dev motion targets: /media/hdd0/whyz/structure-stability/outputs/eda_preprocessing/dev_motion_targets.csv
motion teacher normalization: StandardScaler fitted on train split
train motion rows: 1000 dev motion rows: 100
train: (1000, 10)
dev: (100, 10)


In [3]:
def _extract_frame_by_sec(cap, sec, fps, frame_count):
    frame_idx = int(max(0, min(frame_count - 1, round(sec * fps))))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _extract_last_frame(cap, frame_count):
    last_idx = max(0, frame_count - 1)
    cap.set(cv2.CAP_PROP_POS_FRAMES, last_idx)
    ok, frame = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _video_aug_cache_signature(cfg):
    keys = [
        'SEED',
        'UNSTABLE_START_MIN_SEC',
        'UNSTABLE_START_MAX_SEC',
        'UNSTABLE_FRAMES_MIN',
        'UNSTABLE_FRAMES_MAX',
        'STABLE_END_MIN_SEC',
        'STABLE_END_MAX_SEC',
        'STABLE_FRAMES_PER_VIDEO',
    ]
    return {k: cfg.get(k) for k in keys}


def build_video_augmented_df(train_df, data_dir, cfg):
    train_root = data_dir / 'train'
    aug_root = data_dir / 'train_video_aug'
    aug_root.mkdir(parents=True, exist_ok=True)

    cache_csv = aug_root / 'aug_df.csv'
    cache_meta = aug_root / 'cache_meta.json'
    cache_sig = _video_aug_cache_signature(cfg)

    if cfg.get('VIDEO_AUG_CACHE', True) and cache_csv.exists() and cache_meta.exists():
        try:
            meta = json.loads(cache_meta.read_text())
            if meta.get('signature') == cache_sig:
                cached_df = pd.read_csv(cache_csv)
                if not cached_df.empty:
                    cached_df['sample_dir'] = str(aug_root)
                    return cached_df
        except Exception as exc:
            print('video aug cache read failed:', exc)

    for p in aug_root.glob('AUGV_*'):
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)

    rng = np.random.default_rng(cfg['SEED'])
    saved_idx = 0
    aug_rows = []

    def save_aug(img, label):
        nonlocal saved_idx
        aug_id = f'AUGV_{saved_idx:07d}'
        out_dir = aug_root / aug_id
        out_dir.mkdir(parents=True, exist_ok=True)
        Image.fromarray(img).save(out_dir / 'front.png')
        Image.fromarray(img).save(out_dir / 'top.png')
        row = {'id': aug_id, 'label': label, 'sample_dir': str(aug_root)}
        for col in ALL_AUX_TARGET_COLUMNS:
            row[col] = np.nan
        aug_rows.append(row)
        saved_idx += 1

    for row in tqdm(train_df.itertuples(index=False), total=len(train_df), desc='video aug', dynamic_ncols=True, ascii=True):
        sample_id = row.id
        label = row.label
        video_path = train_root / sample_id / 'simulation.mp4'
        if not video_path.exists():
            continue

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            continue

        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if fps <= 0 or frame_count <= 0:
            cap.release()
            continue

        if label == 'unstable':
            n_frames = int(rng.integers(cfg['UNSTABLE_FRAMES_MIN'], cfg['UNSTABLE_FRAMES_MAX'] + 1))
            secs = rng.uniform(cfg['UNSTABLE_START_MIN_SEC'], cfg['UNSTABLE_START_MAX_SEC'], size=n_frames)
            for sec in secs:
                frame = _extract_frame_by_sec(cap, float(sec), fps, frame_count)
                if frame is not None:
                    save_aug(frame, label)
        else:
            n_frames = int(cfg['STABLE_FRAMES_PER_VIDEO'])
            secs = rng.uniform(cfg['STABLE_END_MIN_SEC'], cfg['STABLE_END_MAX_SEC'], size=n_frames)
            for sec in secs:
                frame = _extract_frame_by_sec(cap, float(sec), fps, frame_count)
                if frame is not None:
                    save_aug(frame, label)

        cap.release()

    aug_df = pd.DataFrame(aug_rows)
    aug_df.to_csv(cache_csv, index=False)
    cache_meta.write_text(json.dumps({'signature': cache_sig}, indent=2))
    return aug_df


class TeacherRegularizationDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sample_id = str(row['id'])
        base_dir = self.root_dir
        if ('sample_dir' in self.df.columns) and pd.notna(row.get('sample_dir', np.nan)):
            base_dir = str(row['sample_dir'])
        folder_path = os.path.join(base_dir, sample_id)

        views = []
        for name in ['front', 'top']:
            img_path = os.path.join(folder_path, f'{name}.png')
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            views.append(image)

        if self.is_test:
            return {'views': views, 'id': sample_id}

        label = self.label_map[row['label']]
        motion_target = row[MOTION_FEATURES].to_numpy(dtype=np.float32)
        onset_target = np.int64(row[ONSET_CLASS_FEATURE]) if pd.notna(row[ONSET_CLASS_FEATURE]) else -1
        image_target = row[IMAGE_STRUCTURE_FEATURES].to_numpy(dtype=np.float32)
        return {
            'views': views,
            'label': torch.tensor(label, dtype=torch.float32),
            'motion_target': torch.tensor(motion_target, dtype=torch.float32),
            'onset_target': torch.tensor(onset_target, dtype=torch.long),
            'image_target': torch.tensor(image_target, dtype=torch.float32),
            'has_teacher': torch.tensor(float(np.isfinite(motion_target).all() and np.isfinite(image_target).all()), dtype=torch.float32),
            'id': sample_id,
        }


train_transform, test_transform = build_default_transforms(CFG['IMG_SIZE'])
train_df_for_train = train_df.copy()
train_df_for_train['sample_dir'] = str(DATA_DIR / 'train')
if CFG['VIDEO_AUG_ENABLE']:
    aug_df = build_video_augmented_df(train_df, DATA_DIR, CFG)
    if not aug_df.empty:
        train_df_for_train = pd.concat([train_df_for_train, aug_df], ignore_index=True)

dev_domain_df = dev_df.copy()
dev_domain_df['sample_dir'] = str(DATA_DIR / 'dev')
dev_eval_df = dev_df.copy()
dev_eval_df['sample_dir'] = str(DATA_DIR / 'dev')

train_dataset = TeacherRegularizationDataset(train_df_for_train, str(DATA_DIR / 'train'), train_transform, is_test=False)
dev_domain_dataset = TeacherRegularizationDataset(dev_domain_df, str(DATA_DIR / 'dev'), train_transform, is_test=False)
dev_eval_dataset = TeacherRegularizationDataset(dev_eval_df, str(DATA_DIR / 'dev'), test_transform, is_test=False)
test_dataset = TeacherRegularizationDataset(test_df, str(DATA_DIR / 'test'), test_transform, is_test=True)

loader_kwargs = dict(
    batch_size=CFG['BATCH_SIZE'],
    num_workers=CFG['NUM_WORKERS'],
    pin_memory=(device.type == 'cuda'),
)
train_loader = DataLoader(train_dataset, shuffle=True, worker_init_fn=seed_worker, generator=make_generator(CFG['SEED']), **loader_kwargs)
dev_domain_loader = DataLoader(dev_domain_dataset, shuffle=True, worker_init_fn=seed_worker, generator=make_generator(CFG['SEED'] + 1), **loader_kwargs)
dev_eval_loader = DataLoader(dev_eval_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print('train_dataset:', len(train_dataset))
print('dev_domain_dataset:', len(dev_domain_dataset))
print('dev_eval_dataset:', len(dev_eval_dataset))
print('test_dataset:', len(test_dataset))


train_dataset: 3000
dev_domain_dataset: 100
dev_eval_dataset: 100
test_dataset: 1000


In [4]:
@dataclass(frozen=True)
class TeacherRegularizationConfig:
    backbone_name: str = CFG['BACKBONE_NAME']
    pretrained: bool = True
    attn_dim: int = CFG['ATTN_DIM']
    num_heads: int = CFG['NUM_HEADS']
    num_layers: int = CFG['NUM_LAYERS']
    pos_grid: int = CFG['POS_GRID']
    dropout: float = CFG['DROPOUT']
    classifier_hidden_dim: int = CFG['CLASSIFIER_HIDDEN_DIM']
    classifier_mid_dim: int = CFG['CLASSIFIER_MID_DIM']
    classifier_dropout: float = CFG['CLASSIFIER_DROPOUT']
    domain_hidden_dim: int = CFG['DOMAIN_HIDDEN_DIM']
    domain_dropout: float = CFG['DOMAIN_DROPOUT']
    motion_dim: int = len(MOTION_FEATURES)
    onset_dim: int = 4
    image_dim: int = len(IMAGE_STRUCTURE_FEATURES)
    out_dim: int = 1


class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None


def grad_reverse(x, lambda_=1.0):
    return GradientReversalFunction.apply(x, lambda_)


class CrossAttentionBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        self.norm_q = nn.LayerNorm(dim)
        self.norm_kv = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)

    def forward(self, q_tokens, kv_tokens):
        q = self.norm_q(q_tokens)
        kv = self.norm_kv(kv_tokens)
        attn_out, _ = self.attn(q, kv, kv, need_weights=False)
        return q_tokens + attn_out


class TeacherRegularizedMultiView(nn.Module):
    def __init__(self, config: TeacherRegularizationConfig | None = None):
        super().__init__()
        self.config = config or TeacherRegularizationConfig()
        self.backbone = timm.create_model(
            self.config.backbone_name,
            pretrained=self.config.pretrained,
            num_classes=0,
            global_pool='',
        )
        feature_dim = self.backbone.num_features
        self.proj = nn.Conv2d(feature_dim, self.config.attn_dim, kernel_size=1, bias=False)
        self.pos_embed = nn.Parameter(torch.randn(1, self.config.attn_dim, self.config.pos_grid, self.config.pos_grid) * 0.02)
        self.cross_12 = nn.ModuleList([CrossAttentionBlock(self.config.attn_dim, self.config.num_heads, self.config.dropout) for _ in range(self.config.num_layers)])
        self.cross_21 = nn.ModuleList([CrossAttentionBlock(self.config.attn_dim, self.config.num_heads, self.config.dropout) for _ in range(self.config.num_layers)])
        self.classifier = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.classifier_hidden_dim),
            nn.BatchNorm1d(self.config.classifier_hidden_dim),
            nn.ReLU(),
            nn.Dropout(self.config.classifier_dropout),
            nn.Linear(self.config.classifier_hidden_dim, self.config.classifier_mid_dim),
            nn.ReLU(),
            nn.Linear(self.config.classifier_mid_dim, self.config.out_dim),
        )
        self.motion_head = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.classifier_hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(self.config.classifier_hidden_dim // 2, self.config.motion_dim),
        )
        self.onset_head = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.classifier_hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(self.config.classifier_hidden_dim // 2, self.config.onset_dim),
        )
        self.image_head = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.classifier_hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(self.config.classifier_hidden_dim // 2, self.config.image_dim),
        )
        self.domain_head = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.domain_hidden_dim),
            nn.ReLU(),
            nn.Dropout(self.config.domain_dropout),
            nn.Linear(self.config.domain_hidden_dim, 1),
        )

    def _to_tokens(self, feat_map):
        feat_map = self.proj(feat_map)
        pos = F.interpolate(self.pos_embed, size=feat_map.shape[-2:], mode='bilinear', align_corners=False)
        feat_map = feat_map + pos
        return feat_map.flatten(2).transpose(1, 2)

    def extract_features(self, views):
        f1 = self.backbone.forward_features(views[0])
        f2 = self.backbone.forward_features(views[1])
        t1 = self._to_tokens(f1)
        t2 = self._to_tokens(f2)
        for blk12, blk21 in zip(self.cross_12, self.cross_21):
            t1_prev, t2_prev = t1, t2
            t1 = blk12(t1_prev, t2_prev)
            t2 = blk21(t2_prev, t1_prev)
        return torch.cat([t1.mean(dim=1), t2.mean(dim=1)], dim=1)

    def forward(self, views, lambda_=0.0):
        feat = self.extract_features(views)
        out = {
            'class_logit': self.classifier(feat).view(-1),
            'motion_pred': self.motion_head(feat),
            'onset_logit': self.onset_head(feat),
            'image_pred': self.image_head(feat),
            'feat': feat,
        }
        dom_feat = grad_reverse(feat, lambda_)
        out['domain_logit'] = self.domain_head(dom_feat).view(-1)
        return out


In [5]:
def mixup_multiview_batch(views, labels, alpha=0.2):
    if alpha <= 0:
        return views, labels, labels, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = labels.size(0)
    index = torch.randperm(batch_size, device=labels.device)
    mixed_views = [lam * v + (1.0 - lam) * v[index, :] for v in views]
    return mixed_views, labels, labels[index], lam


def compute_lambda(epoch_idx, step_idx, steps_per_epoch, total_epochs, warmup_epochs=0, max_lambda=1.0, gamma=10.0):
    total_steps = max(1, total_epochs * steps_per_epoch)
    current_step = max(0, (epoch_idx - 1) * steps_per_epoch + step_idx)
    progress = current_step / total_steps
    warmup_progress = warmup_epochs / max(1, total_epochs)
    if progress <= warmup_progress:
        return 0.0
    effective_progress = (progress - warmup_progress) / max(1e-8, 1.0 - warmup_progress)
    lambda_base = 2.0 / (1.0 + np.exp(-gamma * effective_progress)) - 1.0
    return float(max_lambda * lambda_base)


def masked_smooth_l1(pred, target):
    mask = torch.isfinite(target).all(dim=1)
    if mask.any():
        return F.smooth_l1_loss(pred[mask], target[mask])
    return pred.sum() * 0.0


def masked_cross_entropy(logit, target):
    mask = target >= 0
    if mask.any():
        return F.cross_entropy(logit[mask], target[mask])
    return logit.sum() * 0.0


def train_one_epoch(model, train_loader, dev_loader, optimizer, device, epoch_idx, total_epochs, ema=None):
    model.train()
    dev_iter = cycle(dev_loader)
    total_rows = []
    steps = len(train_loader)
    for step_idx, batch in enumerate(tqdm(train_loader, desc='Training', dynamic_ncols=True, ascii=True), start=1):
        dev_batch = next(dev_iter)
        train_views = [v.to(device) for v in batch['views']]
        train_labels = batch['label'].to(device).float()
        train_motion = batch['motion_target'].to(device)
        train_onset = batch['onset_target'].to(device)
        train_img = batch['image_target'].to(device)
        dev_views = [v.to(device) for v in dev_batch['views']]

        optimizer.zero_grad()

        if CFG['MIXUP_ALPHA'] > 0 and np.random.rand() < CFG['MIXUP_PROB']:
            mixed_views, labels_a, labels_b, lam = mixup_multiview_batch(train_views, train_labels, CFG['MIXUP_ALPHA'])
            outputs = model(mixed_views, lambda_=0.0)
            loss_cls = lam * F.binary_cross_entropy_with_logits(outputs['class_logit'], labels_a) + (1.0 - lam) * F.binary_cross_entropy_with_logits(outputs['class_logit'], labels_b)
            loss_motion = outputs['motion_pred'].sum() * 0.0
            loss_onset = outputs['onset_logit'].sum() * 0.0
            loss_img = outputs['image_pred'].sum() * 0.0
        else:
            outputs = model(train_views, lambda_=0.0)
            loss_cls = F.binary_cross_entropy_with_logits(outputs['class_logit'], train_labels)
            loss_motion = masked_smooth_l1(outputs['motion_pred'], train_motion)
            loss_onset = masked_cross_entropy(outputs['onset_logit'], train_onset)
            loss_img = masked_smooth_l1(outputs['image_pred'], train_img)

        domain_views = [torch.cat([tv, dv], dim=0) for tv, dv in zip(train_views, dev_views)]
        domain_labels = torch.cat([
            torch.zeros(train_views[0].size(0), device=device),
            torch.ones(dev_views[0].size(0), device=device),
        ], dim=0)
        grl_lambda = compute_lambda(
            epoch_idx,
            step_idx - 1,
            steps,
            total_epochs,
            warmup_epochs=CFG['GRL_WARMUP_EPOCHS'],
            max_lambda=CFG['GRL_MAX_LAMBDA'],
            gamma=CFG['GRL_GAMMA'],
        )
        domain_outputs = model(domain_views, lambda_=grl_lambda)
        loss_dom = F.binary_cross_entropy_with_logits(domain_outputs['domain_logit'], domain_labels)

        norm_onset = loss_onset / np.log(4.0)
        loss = (
            loss_cls
            + CFG['MOTION_LOSS_WEIGHT'] * loss_motion
            + CFG['ONSET_LOSS_WEIGHT'] * norm_onset
            + CFG['IMAGE_LOSS_WEIGHT'] * loss_img
            + CFG['DOMAIN_LOSS_WEIGHT'] * loss_dom
        )
        loss.backward()
        optimizer.step()
        if ema is not None:
            ema.update(model)

        with torch.no_grad():
            domain_probs = torch.sigmoid(domain_outputs['domain_logit'])
            domain_acc = ((domain_probs > 0.5) == domain_labels).float().mean().item()

        total_rows.append({
            'loss': float(loss.item()),
            'loss_cls': float(loss_cls.item()),
            'loss_motion': float(loss_motion.item()),
            'loss_onset': float(loss_onset.item()),
            'loss_img': float(loss_img.item()),
            'loss_dom': float(loss_dom.item()),
            'domain_acc': float(domain_acc),
        })

    hist = pd.DataFrame(total_rows)
    return hist.mean().to_dict()


@torch.no_grad()
def evaluate_classification(model, loader, device):
    model.eval()
    all_probs = []
    all_labels = []
    motion_losses = []
    onset_losses = []
    image_losses = []
    for batch in tqdm(loader, desc='Dev Eval', dynamic_ncols=True, ascii=True):
        views = [v.to(device) for v in batch['views']]
        labels = batch['label'].to(device).float()
        motion_target = batch['motion_target'].to(device)
        onset_target = batch['onset_target'].to(device)
        image_target = batch['image_target'].to(device)
        outputs = model(views, lambda_=0.0)
        probs = torch.sigmoid(outputs['class_logit'])
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        if bool(torch.isfinite(motion_target).all(dim=1).any()):
            motion_losses.append(float(masked_smooth_l1(outputs['motion_pred'], motion_target).item()))
        if bool((onset_target >= 0).any()):
            onset_losses.append(float(masked_cross_entropy(outputs['onset_logit'], onset_target).item()))
        if bool(torch.isfinite(image_target).all(dim=1).any()):
            image_losses.append(float(masked_smooth_l1(outputs['image_pred'], image_target).item()))

    all_probs = np.array(all_probs, dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)
    p = np.clip(all_probs, 1e-15, 1 - 1e-15)
    logloss = float(-np.mean(all_labels * np.log(p) + (1.0 - all_labels) * np.log(1.0 - p)))
    acc = float(np.mean((all_probs > 0.5) == all_labels))
    auc = float(__import__('sklearn.metrics').metrics.roc_auc_score(all_labels, all_probs))
    return {
        'dev_logloss': logloss,
        'dev_acc': acc,
        'dev_auc': auc,
        'dev_motion_loss': float(np.mean(motion_losses)) if motion_losses else float('nan'),
        'dev_onset_loss': float(np.mean(onset_losses)) if onset_losses else float('nan'),
        'dev_img_loss': float(np.mean(image_losses)) if image_losses else float('nan'),
    }


@torch.no_grad()
def predict_test_probs(model, loader, device):
    model.eval()
    probs_all = []
    ids = []
    for batch in tqdm(loader, desc='Test Inference', dynamic_ncols=True, ascii=True):
        views = [v.to(device) for v in batch['views']]
        outputs = model(views, lambda_=0.0)
        probs = torch.sigmoid(outputs['class_logit'])
        probs_all.extend(probs.cpu().numpy())
        ids.extend(batch['id'])
    return ids, np.array(probs_all, dtype=np.float64)


In [6]:

model = TeacherRegularizedMultiView(TeacherRegularizationConfig()).to(device)
optimizer = optim.Adam(model.parameters(), lr=CFG['LEARNING_RATE'], weight_decay=CFG['WEIGHT_DECAY'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['EPOCHS'], eta_min=CFG['MIN_LR'])
ema = ModelEMA(model, EMAConfig(decay=CFG['EMA_DECAY']))

artifacts = allocate_output_paths(experiment_name='teacher_regularization', major_version='v1.2')
best_model_path = artifacts['weight_path']
submission_path = artifacts['submission_path']
history_path = (Path.cwd() / '../../outputs/eda_preprocessing/teacher_regularization_v1.2.4_history.csv').resolve()
history_path.parent.mkdir(parents=True, exist_ok=True)
print('Artifact version:', artifacts['version'])
print('best_model_path:', best_model_path)
print('submission_path:', submission_path)

best_logloss = float('inf')
best_epoch = -1
patience_left = CFG['EARLY_STOPPING_PATIENCE']
history = []

for epoch in range(1, CFG['EPOCHS'] + 1):
    train_metrics = train_one_epoch(model, train_loader, dev_domain_loader, optimizer, device, epoch, CFG['EPOCHS'], ema=ema)
    eval_model = ema.ema_model if CFG['EMA_USE_FOR_EVAL'] else model
    dev_metrics = evaluate_classification(eval_model, dev_eval_loader, device)

    improved = dev_metrics['dev_logloss'] < best_logloss
    if improved:
        best_logloss = dev_metrics['dev_logloss']
        best_epoch = epoch
        patience_left = CFG['EARLY_STOPPING_PATIENCE']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'ema_state_dict': ema.ema_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'cfg': CFG,
            **dev_metrics,
        }, best_model_path)
    else:
        patience_left -= 1

    scheduler.step()
    row = {
        'epoch': epoch,
        **{k: float(v) for k, v in train_metrics.items()},
        **dev_metrics,
        'lr': float(optimizer.param_groups[0]['lr']),
        'improved': bool(improved),
        'patience_left': int(patience_left),
    }
    history.append(row)
    print(row)
    if patience_left < 0:
        print('early stop:', epoch)
        break

history_df = pd.DataFrame(history)
history_df.to_csv(history_path, index=False)
print('saved history:', history_path)


Artifact version: v1.2.0
best_model_path: /media/hdd0/whyz/structure-stability/notebooks/outputs/weights/teacher_regularization_v1.2.0.pt
submission_path: /media/hdd0/whyz/structure-stability/notebooks/outputs/submissions/teacher_regularization_v1.2.0.csv


Training:   0%|          | 0/94 [00:02<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 294.00 MiB. GPU 0 has a total capacity of 23.55 GiB of which 157.38 MiB is free. Process 4335 has 260.00 MiB memory in use. Including non-PyTorch memory, this process has 22.93 GiB memory in use. Of the allocated memory 21.96 GiB is allocated by PyTorch, and 657.04 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:

checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
best_state = checkpoint['ema_state_dict'] if CFG['EMA_USE_FOR_EVAL'] else checkpoint['model_state_dict']
model.load_state_dict(best_state)
final_dev_metrics = evaluate_classification(model, dev_eval_loader, device)
print('best_epoch:', best_epoch)
print(final_dev_metrics)

ids, test_probs = predict_test_probs(model, test_loader, device)
submission = pd.DataFrame({
    'id': ids,
    'unstable_prob': test_probs,
    'stable_prob': 1.0 - test_probs,
})
submission.to_csv(submission_path, index=False, encoding='utf-8-sig')
print('saved submission:', submission_path)
submission.head()


Dev Eval: 100%|##########| 13/13 [00:00<00:00, 14.65it/s]


best_epoch: 7
{'dev_logloss': 0.07854000844858816, 'dev_acc': 0.99, 'dev_auc': 1.0, 'dev_phys_loss': nan, 'dev_img_loss': 0.71633403805586}


Test Inference: 100%|##########| 125/125 [00:05<00:00, 23.70it/s]

saved submission: /media/hdd0/whyz/structure-stability/outputs/submissions/teacher_regularization_v1.4.0.csv


,id,unstable_prob,stable_prob
0,TEST_0001,0.078139,0.921861
1,TEST_0002,0.994145,0.005855
2,TEST_0003,0.995798,0.004202
3,TEST_0004,0.991625,0.008375
4,TEST_0005,0.041346,0.958654


: 